In [1]:
import pandas as pd
import torch
import os
import numpy as np
from PIL import Image
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from nltk.tokenize import word_tokenize
from collections import Counter
import nltk
nltk.download("punkt_tab")
torch.manual_seed(88)

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/tinotendanyamutwa/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [2]:
#load data
train_pairs = pd.read_csv("train_pairs.csv.gz")
val_pairs = pd.read_csv("val_pairs.csv.gz")

In [3]:
all_tokens = []
for descr in train_pairs["description"]:
    tokens = word_tokenize(descr.lower())
    all_tokens.extend(tokens)

cnt = Counter(all_tokens)
top = cnt.most_common(10000)

vocab = {"<pad>": 0, "<unk>": 1}
for i, (word, count) in enumerate(top):
    vocab[word] = i + 2

print(len(vocab))

10002


In [4]:
#encoder handles each tensor 
class encode_image(nn.Module):
    #the existing layers 
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2)
        self.flatten = nn.Flatten()
        self.linear = nn.Linear(64 * 8 * 8, 128)
    #the order the layers work 
    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = self.pool(self.relu(self.conv3(x)))
        x = self.flatten(x)
        x = self.linear(x)
        return x

In [5]:
#splits text into words stores as lowercase tokens 
#<unk> (1) are for words not in vocab
#keeps first 128 ids 
#less than 128 ids then rest are padded 
#turns list into pyTorch tensor
#long makes it int64
def encode_text(s):
    tokens = word_tokenize(s.lower())
    ids = []
    for word in tokens:
        ids.append(vocab.get(word,1))
    ids = ids[:128] 
    while len(ids) < 128:
        ids.append(0)
    return torch.tensor(ids, dtype=torch.long)

In [6]:
#where the ids are turned into 128 number vector 
class TextEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(len(vocab), 128)
        self.fc = nn.Linear(128, 128)
        
    def forward(self, x):
        #turns ids into 128 number vector
        x = self.embedding(x)
        #the average across the tokens 
        #128 vector summarising the whole description
        x = torch.mean(x, dim=1)
        #128 vectors in 128 out and the weights are learned 
        x = self.fc(x)
        return x

In [7]:
#where image features and text features become one vector
#and decideds if they are match or not match
class Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(256, 128)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, 1)

    def forward(self, img, txt):
        #the vectors are glued end to end to make batch, 256
        x = torch.cat([img, txt], dim=1)
        #256 batch vector becomes 128 batch vector
        x = self.fc1(x)
        #non linearity stuff
        x = self.relu(x)
        #theres one number per sample
        x = self.fc2(x)
        return x

In [8]:
class CombinedDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, i):
        img_path = self.pairs["image_path"].iloc[i]
        label = self.pairs["label"].iloc[i]
        descr = self.pairs["description"].iloc[i]

        img = Image.open(img_path)
        img = img.convert("RGB")
        img = img.resize((64, 64)) 
        arr = np.array(img) / 255.0
        img_tensor = torch.tensor(arr).permute(2, 0, 1).float()

        ids_tensor = encode_text(descr)
        label_tensor = torch.tensor(label, dtype=torch.float32)

        return img_tensor, ids_tensor, label_tensor

In [9]:
#store the data 
dataset = CombinedDataset(train_pairs)
dataset2 = CombinedDataset(val_pairs)

In [10]:
#get instances of the endcoders and classifier
img_encoder = encode_image()
txt_encoder = TextEncoder()
classifier = Classifier()

#collects all the learnable wights from encoders and classifiers
params = list(img_encoder.parameters()) + list(txt_encoder.parameters()) + list(classifier.parameters())
optimizer = torch.optim.Adam(params, lr=1e-3)
loss_fn = nn.BCEWithLogitsLoss()

In [11]:
#learning loop trains for 3 epochs, validates epoch, then prints the progress
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(dataset2, batch_size=64, shuffle=False)

epochs = 3

for epoch in range(epochs):
    #for one epock
    #in training pass the training loop for each batch predicts measures the loss, ajust the weights
    #done until all data seen
    #for val pass it predicts measures loss but doesnt adjust weights just measures the loss
    #both averages are printed
    img_encoder.train(); txt_encoder.train(); classifier.train()
    total_loss = 0
    for imgs, ids, labels in train_loader:
        optimizer.zero_grad()
        img_f = img_encoder(imgs)
        txt_f = txt_encoder(ids)
        logits = classifier(img_f, txt_f).squeeze(1)
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    train_loss = total_loss / len(train_loader)

    img_encoder.eval(); txt_encoder.eval(); classifier.eval()
    val_loss= 0
    with torch.no_grad():
        for imgs, ids, labels in val_loader:
            img_f = img_encoder(imgs)
            txt_f = txt_encoder(ids)
            logits = classifier(img_f, txt_f).squeeze(1)
            loss = loss_fn(logits, labels)
            val_loss += loss.item()
    val_loss = val_loss / len(val_loader)

    print(f"epoch {epoch+1} train {train_loss:.4f} val{val_loss:.4f}")

epoch 1 train 0.6064 val0.4515
epoch 2 train 0.4099 val0.3767
epoch 3 train 0.3588 val0.3663


In [12]:
torch.save({
    "img_encoder": img_encoder.state_dict(),
    "txt_encoder": txt_encoder.state_dict(),
    "classifier": classifier.state_dict(),
    "vocab": vocab,
}, "baseline_checkpoints.pt")

In [13]:
import json
with open("vocab.json", "w") as f:
    json.dump(vocab,f)

In [14]:
#a loadable file that proves the model works to see real predictions
ckpt = torch.load("baseline_checkpoints.pt")
img_encoder.load_state_dict(ckpt["img_encoder"])
txt_encoder.load_state_dict(ckpt["txt_encoder"])
classifier.load_state_dict(ckpt["classifier"])
assert ckpt["vocab"] == vocab
print("loaded ok")

img_encoder.eval(); txt_encoder.eval(); classifier.eval()
imgs, ids, labels = next(iter(val_loader))
with torch.no_grad():
    logits = classifier(img_encoder(imgs), txt_encoder(ids)).squeeze(1)
    preds = (logits > 0).float()
    print("acc:", (preds == labels).float().mean().item())

loaded ok
acc: 0.78125
